In [48]:
import pandas as pd

df = pd.read_csv('../Planilhas/imoveis_unificado.csv', encoding='utf-8', sep=';')

C:\Users\danie\AppData\Local\Temp\ipykernel_24352\869521274.py:3: DtypeWarning: Columns (0: numero, 1: ano_construcao, 2: cod_logradouro, 3: latitude, 4: longitude, 5: ano, 6: distrito) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Planilhas/imoveis_unificado.csv', encoding='utf-8', sep=';')


---

## **Coluna Número**

### Diagnóstico da coluna

Antes de qualquer transformação, é essencial entender o estado real dos dados. Vamos verificar os tipos, contar nulos e olhar os valores mais frequentes para saber o que precisamos tratar.

In [9]:
# visão geral da coluna
print("Tipo:", df['numero'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['numero'].isna().sum())
print("Valores únicos:", df['numero'].nunique())
print()
print("Amostra dos 20 valores mais frequentes:")
print(df['numero'].value_counts(dropna=False).head(20))

Tipo: object
Total de linhas: 44234
Nulos: 1446
Valores únicos: 13944

Amostra dos 20 valores mais frequentes:
numero
NaN     1446
179      398
1069     357
251      340
1000     307
80       296
56       253
300      242
16       237
1280     235
141      214
215      185
633      174
100      172
1297     167
234      154
121      148
1000     146
60       138
160      132
Name: count, dtype: int64


### Passo 1 - converter tudo para string

A coluna `numero` é do tipo `object`, o que significa que o pandas
está misturando tipos diferentes na mesma coluna: algumas células
podem ser inteiros (`123`), outras floats (`123.0`), outras strings
(`"123"`). Isso causa erros nas etapas de limpeza seguintes.

Para resolver, convertemos tudo para `str` com `.astype(str)`.
O problema é que essa conversão também transforma os valores nulos
(`NaN`) na string `"nan"` — que não é um nulo de verdade, é só um
texto. Por isso logo em seguida substituímos `"nan"` de volta para
`NaN` real, preservando a informação de que aquela célula estava vazia.

In [10]:
import numpy as np

# converte para string
df['numero'] = df['numero'].astype(str)

# "nan" virou string na conversão, voltamos para NaN real
df['numero'] = df['numero'].replace('nan', np.nan)

### Passo 2 - tratar o valor `-1`

O valor `-1` é um sentinela usado para indicar "sem número" (imóveis do tipo S/N). Precisamos padronizá-lo para `"SN"` junto com outras variantes comuns do mesmo significado.

- -1
- "S/N", "s/n", "SEM NUMERO"
- 0

In [11]:
# remove espaços e coloca em maiúsculo para comparação segura
df['numero'] = df['numero'].str.strip().str.upper()

# todas as variantes de "sem número" viram "SN"
sem_numero = ['-1', '0', 'S/N', 'SEM NUMERO',
              'SEM NÚMERO', 'S.N.', 'S.N', 'N/A']

df['numero'] = df['numero'].replace(sem_numero, 'SN')

### Passo 3 - limpar caracteres indesejados

Números de endereço podem ter sujeiras como espaços extras e zeros à
esquerda (`"007"` → `"7"`). Além disso, sufixos de complemento podem
aparecer com ou sem hífen (`"123A"` e `"123-A"` representam o mesmo
endereço). Vamos padronizar removendo zeros à esquerda e garantindo
que o hífen sempre esteja presente entre o número e o sufixo.

In [12]:
import re

def limpar_numero(val):
    if pd.isna(val) or val == 'SN':
        return val
    # remove tudo exceto dígitos, letras e hífen
    val = re.sub(r'[^\w\-]', '', val)
    # remove zeros à esquerda em partes numéricas
    val = re.sub(r'\b0+(\d)', r'\1', val)
    # padroniza sufixo: "123A" → "123-A"
    val = re.sub(r'(\d)([A-Z])', r'\1-\2', val)
    
    return val.strip() or None

df['numero'] = df['numero'].apply(limpar_numero)

### Passo 4 - tratar nulos restantes

Após as etapas anteriores ainda podem restar `NaN`. Para imóveis sem número cadastrado o comportamento mais seguro é preencher com `"SN"`, pois é a forma padrão de registrar a ausência de numeração.

In [13]:
# preenche nulos restantes com "SN"
df['numero'] = df['numero'].fillna('SN')

print("Nulos após tratamento:", df['numero'].isna().sum())

Nulos após tratamento: 0


### Verificação do resultado

Conferimos se o tratamento foi aplicado corretamente: não deve restar nenhum `-1`, zero, ou variante de S/N na coluna.

In [14]:
print("=== Resultado final ===")
print("Tipo da coluna:", df['numero'].dtype)
print("Nulos:", df['numero'].isna().sum())
print("Contagem de SN:", (df['numero'] == 'SN').sum())
print()

# checar se ainda existe algum sentinela
sentinelas_restantes = df['numero'].isin(['-1', '0', 'S/N']).sum()
print(f"Sentinelas restantes: {sentinelas_restantes}")
print()
print("Top 10 valores mais frequentes:")
print(df['numero'].value_counts().head(10))

=== Resultado final ===
Tipo da coluna: str
Nulos: 0
Contagem de SN: 1460

Sentinelas restantes: 0

Top 10 valores mais frequentes:
numero
SN      1460
1000     453
251      453
179      415
1069     357
300      333
80       328
56       278
100      263
121      248
Name: count, dtype: int64


---

## **Coluna Latitude**

### Diagnóstico da coluna

Antes de qualquer decisão, precisamos entender o estado real da coluna: quantos nulos existem, qual o tipo atual, e se os valores fazem sentido geograficamente para o Recife.

In [15]:
print("Tipo:", df['latitude'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['latitude'].isna().sum())
print(f"Percentual de nulos: {df['latitude'].isna().mean() * 100:.1f}%")
print()
print(df['latitude'].describe())

Tipo: object
Total de linhas: 44234
Nulos: 14874
Percentual de nulos: 33.6%

count         29360
unique        10413
top       -8.085739
freq            246
Name: latitude, dtype: object


### Decisão: remover ou manter a coluna?

A `latitude` é uma informação geográfica que complementa o endereço, mas a base do ITBI já conta com outras colunas de localização (logradouro, bairro, etc.). Por isso ela pode ser considerada redundante.

Considerando que o percentual de nulos é alto (acima de 30%), o custo de imputar ou ignorar os nulos supera o benefício da coluna, logo foi decidido que o ideal é removê-la.

In [16]:
df = df.drop(columns=['latitude'])
print("Coluna 'latitude' removida.")
print("Colunas restantes:", df.columns.tolist())

Coluna 'latitude' removida.
Colunas restantes: ['logradouro', 'numero', 'complemento', 'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao', 'area_terreno', 'area_construida', 'fracao_ideal', 'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao', 'data_transacao', 'estado_conservacao', 'tipo_imovel', 'sfh', 'cod_logradouro', 'longitude', 'ano', 'distrito']


---

## **Coluna Longitude**

### Diagnóstico da coluna

É esperado que a `longitude` siga o mesmo caminho da `latitude`, mas é importante entender o estado real da coluna antes de tomar qualquer decisão precipitada

In [ ]:
print("Tipo:", df['longitude'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['longitude'].isna().sum())
print(f"Percentual de nulos: {df['longitude'].isna().mean() * 100:.1f}%")
print()
print(df['longitude'].describe())

Tipo: object
Total de linhas: 44234
Nulos: 9253
Percentual de nulos: 20.9%

count     34981
unique     6645
top        2024
freq      15243
Name: longitude, dtype: object
0


Seguindo a mesma lógica aplicada na coluna `latitude`, foi decidido pela remoção da coluna `longitude`.

In [ ]:
df = df.drop(columns=['longitude'])
print("Coluna 'longitude' removida.")
print("Colunas restantes:", df.columns.tolist())

---

## **Coluna Fração Ideal**

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [49]:
print("Tipo:", df['fracao_ideal'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['fracao_ideal'].isna().sum())
print("Valores únicos:", df['fracao_ideal'].nunique())
print()
print("20 valores mais frequentes:")
print(df['fracao_ideal'].value_counts(dropna=False).head(20))

Tipo: str
Total de linhas: 44234
Nulos: 0
Valores únicos: 4653

20 valores mais frequentes:
fracao_ideal
1,0        2721
1          1352
0,16666     356
0,0014      338
0,03125     271
0,00001     264
0,00199     263
0,04166     260
0,002       254
0,08333     253
0,00417     246
0,00446     238
0,00138     226
0,0024      215
0,0025      195
0,01785     194
0,03333     193
0,02272     192
0,025       190
0,03846     189
Name: count, dtype: int64


### Passo 1 - padronizar o formato decimal

Como os valores podem vir com vírgula decimal (`0,16666`), vamos converter para o padrão usado pelo Python (`0.16666`).

In [50]:
df['fracao_ideal'] = (
    df['fracao_ideal']
    .astype(str)
    .str.strip()
    .str.replace(',', '.', regex=False)
)

### Passo 2 - testar a conversão para número

Agora tentamos converter toda a coluna para `float`.

Qualquer valor que não seja numérico será transformado em NaN, permitindo identificar problemas facilmente.

In [51]:
temp = pd.to_numeric(
    df['fracao_ideal'],
    errors='coerce'
)

print("Valores que não puderam ser convertidos:",
      temp.isna().sum())

Valores que não puderam ser convertidos: 3


Agora que sabemos que todos os valores são numéricos, podemos converter a coluna.

In [52]:
df['fracao_ideal'] = pd.to_numeric(
    df['fracao_ideal'],
    errors='coerce'
)

### Passo 3 - verificação do resultado

Conferimos se o tratamento foi aplicado corretamente: não deve restar nenhum valor nulo na coluna e o tipo deve ser `float`.

In [53]:
print("Tipo:", df['fracao_ideal'].dtype)
print("Nulos:", df['fracao_ideal'].isna().sum())
print()

print(df['fracao_ideal'].describe())

Tipo: float64
Nulos: 3

count    44231.000000
mean         0.114610
std          0.286649
min          0.000000
25%          0.004660
50%          0.013150
75%          0.031315
max          1.180050
Name: fracao_ideal, dtype: float64


### Passo 4 - checagem de números maiores que 1 (mais de 100%)

Visto que é uma fração que o intervalo esperado seria de 0-100% é necessário analisar quais casos estão ultrapassando o 100%.

In [39]:
print("Quantidade > 1:", (df['fracao_ideal'] > 1).sum())
print()
df[df['fracao_ideal'] > 1]

Quantidade > 1: 2



,logradouro,numero,complemento,valor_avaliacao,bairro,cidade,uf,ano_construcao,area_terreno,area_construida,...,tipo_construcao,tipo_ocupacao,data_transacao,estado_conservacao,tipo_imovel,sfh,cod_logradouro,longitude,ano,distrito
4413,rua professor ageu magalhaes,211,NaN,1200000.00,Parnamirim,Recife,PE,1997,"254,35","236,36",...,Casa,COMERCIAL COM LIXO ORGANICO,2023-01-10,Regular,Casa,960000.00,1198,-34.913381,2023,3
7806,rua aurora cacote,602,NaN,550000.00,Areias,Recife,PE,1968,"200,0","268,46",...,Casa,RESIDENCIAL,2023-09-25,Bom,Casa,0.00,9172,-34.933164,2023,5


Foram identificados dois registros com fração ideal superior a 1 (100%). Como a fração ideal representa uma participação proporcional do imóvel, espera-se que seus valores estejam limitados ao intervalo [0.0, 1.0]. Devido à baixa incidência desses casos e à impossibilidade de confirmar o valor correto, os registros foram mantidos para preservar a integridade dos dados originais.

---

## **Coluna Tipo Ocupação**

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [ ]:
print("Tipo:", df['tipo_ocupacao'].dtype)
print("Nulos:", df['tipo_ocupacao'].isna().sum())
print("Valores únicos:", df['tipo_ocupacao'].nunique())
print()
print(df['tipo_ocupacao'].value_counts(dropna=False).head(20))

Tipo: str
Nulos: 0
Valores únicos: 4

tipo_ocupacao
RESIDENCIAL                    38403
COMERCIAL COM LIXO ORGANICO     3320
COMERCIAL SEM LIXO ORGANICO     2508
tipo_ocupacao                      3
Name: count, dtype: int64


A coluna `tipo_ocupacao` possui natureza categórica e apresenta apenas três categorias distintas. Não foram identificados valores nulos nem inconsistências de preenchimento. Foi definido que a coluna já se encontrava consistente e não demandou tratamentos adicionais.